# Estadística descriptiva de divorcios y delitos
En esta libreta se resumen los indicadores clásicos (media, mediana, desviación estándar, 
mínimo, máximo e IQR) para divorcios y los principales tipos de delito registrados entre 2001 y 2024.
El énfasis está en comparar media vs. mediana para detectar sesgos fuertes que puedan distorsionar
la interpretación central de los eventos.


## Fuentes y preparación
Se utilizan los archivos del directorio `datos/` y se transforman a formato largo para tener las
columnas `Departamento`, `Anio`, `valor` y `dataset`. Se eliminan los totales nacionales para evitar
contar dos veces la misma información.


In [ ]:
import pandas as pd
from pathlib import Path
from typing import Iterable, Optional

DATASETS = {
    "divorcios": "Divorcios 2001 a 2024.csv",
    "delitos_total": "Delitos 2014 a 2024.csv",
    "otros_delitos": "Otros delitos 2014 a 2024.csv",
    "delincuencia": "Proyecto1(Delitos 2014 a 2024 Delincuenci).csv",
    "narcos": "Proyecto1(Delitos 2014 a 2024 NARCOS).csv",
    "ramo_penal": "Proyecto1(Ramo Penal 2014 a 2024).csv",
}

DATASET_LABELS = {
    "divorcios": "Divorcios",
    "delitos_total": "Delitos totales",
    "otros_delitos": "Otros delitos",
    "delincuencia": "Delincuencia común",
    "narcos": "Delitos narcotráfico",
    "ramo_penal": "Ramo penal",
}

pd.options.display.float_format = "{:.2f}".format

def _resolve_data_dir() -> Path:
    """Encuentra la carpeta 'datos' desde diferentes posibles raíces."""
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / "datos"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No se encontró la carpeta 'datos'.")

DATA_DIR = _resolve_data_dir()

def load_long_form(dataset_subset: Optional[Iterable[str]] = None) -> pd.DataFrame:
    """Carga y limpia los datos en formato largo filtrando totales nacionales."""
    selected = (
        DATASETS
        if dataset_subset is None
        else {k: v for k, v in DATASETS.items() if k in set(dataset_subset)}
    )

    tables = []
    for name, filename in selected.items():
        df = pd.read_csv(
            DATA_DIR / filename,
            sep=";",
            thousands=".",
            decimal=",",
            encoding="latin-1",
            engine="python",
        )
        df.columns = df.columns.str.strip()
        df["Departamento"] = df["Departamento"].str.strip()
        year_cols = [col for col in df.columns if col.strip().isdigit()]
        long_df = (
            df.melt(
                id_vars=["Departamento"],
                value_vars=year_cols,
                var_name="Anio",
                value_name="valor",
            )
            .assign(dataset=name)
            .dropna(subset=["valor"])
        )
        long_df["Anio"] = pd.to_numeric(long_df["Anio"], errors="coerce")
        long_df["valor"] = pd.to_numeric(long_df["valor"], errors="coerce")
        tables.append(long_df.dropna(subset=["Anio", "valor"]))

    data = pd.concat(tables, ignore_index=True)
    data["Departamento"] = (
        data["Departamento"]
        .str.replace("pa�s", "país", regex=False)
        .str.replace("pa�", "pa", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    mask_total = data["Departamento"].str.contains("total", case=False, na=False)
    mask_todos = data["Departamento"].str.contains("todos", case=False, na=False)
    return data.loc[~(mask_total | mask_todos)].reset_index(drop=True)

all_data = load_long_form()
all_data["dataset_label"] = all_data["dataset"].map(DATASET_LABELS)
print(f"Filas cargadas: {len(all_data):,}")
all_data.head()


In [ ]:
def descriptive_table(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby("dataset_label")["valor"]
    summary = grouped.agg(
        observaciones="count",
        media="mean",
        mediana="median",
        desviacion="std",
        minimo="min",
        maximo="max",
    )
    q75 = grouped.quantile(0.75)
    q25 = grouped.quantile(0.25)
    summary["IQR"] = q75 - q25
    summary["Media - mediana"] = summary["media"] - summary["mediana"]
    return summary.sort_values("media", ascending=False)

resumen = descriptive_table(all_data)
resumen.round(2)


## Hallazgos clave
- `Otros delitos` presenta la asimetría más marcada: media ≈640 casos frente a una mediana de 92,
  lo que confirma que pocos años/departamentos concentran picos extremos.
- `Delincuencia común` y `ramo penal` también muestran brechas grandes (≈328 y ≈292 unidades entre
  media y mediana), por lo que la media tiende a sobrestimar la situación típica.
- `Divorcios` y `delitos totales` tienen diferencias menores (≈98 y ≈81), pero aún así la mediana es
  más representativa cuando se quiere hablar de un departamento promedio.
- Los IQR entre 115 y 149 indican alta dispersión espacial y temporal, respaldando el uso de
  medidas robustas antes de comparar territorios.
